# 03. Object Storage – MinIO

## Mục đích

Quy trình này kiểm tra và quản lý dữ liệu đã được lưu trữ trên
Object Storage MinIO sau bước Data Ingestion.

Các nội dung thực hiện:
1. Kiểm tra kết nối MinIO.
2. Kiểm tra bucket ecommerce-lakehouse.
3. Kiểm tra cấu trúc object storage.
4. Liệt kê các object đã lưu.
5. Kiểm tra metadata và dung lượng.
6. Đọc thử một object từ MinIO.
7. Kiểm tra tính toàn vẹn dữ liệu.
8. Chuẩn bị dữ liệu cho Bronze Layer.

Lưu ý: Dữ liệu không được upload lại trong quy trình này.
Phần upload đã được thực hiện ở quy trình Data Ingestion.

In [1]:
import os
from io import BytesIO
from pathlib import Path

import pandas as pd
from minio import Minio

In [2]:
# Thiết lập kết nối MinIO
from minio import Minio

client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

print("✓ Connected to MinIO")

✓ Connected to MinIO


In [3]:
# Kiểm tra bucket tồn tại
BUCKET_NAME = "ecommerce-lakehouse"

if client.bucket_exists(BUCKET_NAME):
    print(f"✓ Bucket exists: {BUCKET_NAME}")
else:
    print("✗ Bucket not found")

✓ Bucket exists: ecommerce-lakehouse


In [4]:
# Kiểm tra tổng quan ingestion
objects = list(
    client.list_objects(
        BUCKET_NAME,
        recursive=True
    )
)

print(f"Tổng số object: {len(objects)}")

for obj in objects:
    print(obj.object_name)

Tổng số object: 7
logs/ingestion_log.json
raw/customers.csv
raw/order_items.csv
raw/orders.csv
raw/payments.csv
raw/products.csv
raw/reviews.csv


In [5]:
# Kiểm tra metadata và dung lượng của các object
for obj in objects:
    print(
        f"{obj.object_name} | "
        f"{obj.size / (1024 * 1024):.2f} MB | "
        f"{obj.last_modified}"
    )

total_size = sum(obj.size for obj in objects)

print(
    f"Tổng dung lượng: "
    f"{total_size / (1024 * 1024):.2f} MB"
)    

logs/ingestion_log.json | 0.00 MB | 2026-09-24 10:37:03.647000+00:00
raw/customers.csv | 8.17 MB | 2026-09-24 10:36:59.749000+00:00
raw/order_items.csv | 14.22 MB | 2026-09-24 10:36:59.898000+00:00
raw/orders.csv | 16.60 MB | 2026-09-24 10:36:59.859000+00:00
raw/payments.csv | 5.37 MB | 2026-09-24 10:36:59.962000+00:00
raw/products.csv | 2.67 MB | 2026-09-24 10:36:59.803000+00:00
raw/reviews.csv | 13.31 MB | 2026-09-24 10:36:59.936000+00:00
Tổng dung lượng: 60.34 MB


In [6]:
# Đọc thử orders.csv từ MinIO
from io import BytesIO
import pandas as pd

response = client.get_object(
    BUCKET_NAME,
    "raw/orders.csv"
)

try:
    data = response.read()
finally:
    response.close()
    response.release_conn()

orders = pd.read_csv(BytesIO(data))

print("✓ Đọc orders.csv thành công")
print("Số dòng:", len(orders))
print("Số cột:", len(orders.columns))

orders.head()

✓ Đọc orders.csv thành công
Số dòng: 99441
Số cột: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
layers = [
    "raw/",
    "bronze/",
    "silver/",
    "gold/",
    "logs/"
]

for layer in layers:
    print(f"├── {layer}")

├── raw/
├── bronze/
├── silver/
├── gold/
├── logs/
